# Project Cancer Detection

# Breast Cancer Wisconsin (Disgnostic) Data Set

[Source: UCI](http://archive.ics.uci.edu/ml/datasets/Breast+Cancer+Wisconsin+%28Diagnostic%29)

[Data Set info](http://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.names)

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
col = ['id', 'Clump Thickness', 'Uniformity of Cell Size', 
       'Uniformity of Cell Shape', 'Marginal Adhesion', 
       'Single Epithelial Cell Size', 'Bare Nuclei', 'Bland Chromatin',
       'Normal Nucleoli', 'Mitoses', 'Class']
# ataching the feature names which we declerified in "col" list on top of of the "df" dataset
df = pd.read_csv("breast-cancer-wisconsin.data.csv", names=col, header=None)
df.head()

# Data Pre-processing

In [ ]:
np.where(df.isnull())

apperantly no-where!

In [ ]:
df.info()

we conclude that the 'Bare Nuclei' feature have some type which did not index as normal data-types

In [ ]:
df['Bare Nuclei'].describe()

In [ ]:
df['Bare Nuclei'].unique()

In [ ]:
df['Bare Nuclei'].value_counts()

How do we drop the `?`

In [ ]:
# the culprit is '?' guy 
df[df['Bare Nuclei'] == "?"]

In [ ]:
df['Class'].value_counts()

In [ ]:
# inplace replacement of  freaky "?" with NaN, in order to being able to drop these data points, using pandas.dropna()
df['Bare Nuclei'].replace("?", np.NAN, inplace=True)
df = df.dropna()

Note that for "class" feature values : 2 is benign, 4 is for malignant (refer to two cells above)

$$\frac{\text{df["Class"]}}{2} - 1$$

In [ ]:
df['Bare Nuclei'].value_counts()

In [ ]:
df['Class'] = df['Class'] / 2 - 1

In [ ]:
df['Class'].value_counts()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
X = df.drop(['id', 'Class'], axis=1)
X_col = X.columns

In [ ]:
y = df['Class']

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X = StandardScaler().fit_transform(X.values)

## Training

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df1 = pd.DataFrame(X, columns=X_col)

In [ ]:
df1.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df1, y,
                                                    train_size=0.8,
                                                    random_state=42)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
pd.DataFrame(MinMaxScaler().fit_transform(df.drop(['id', 'Class'], axis=1).values), columns=X_col).head()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5,
                           p=2, metric='minkowski')

In [ ]:
knn.fit(X_train, y_train)

In [ ]:
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
def print_score(clf, X_train, y_train, X_test, y_test, train=True):
    if train:
        print("Train Result:\n")
        print("accuracy score: {0:.4f}\n".format(accuracy_score(y_train, clf.predict(X_train))))
        print("Classification Report: \n {}\n".format(classification_report(y_train, clf.predict(X_train))))
        print("Confusion Matrix: \n {}\n".format(confusion_matrix(y_train, clf.predict(X_train))))

        res = cross_val_score(clf, X_train, y_train, cv=10, scoring='accuracy')
        print("Average Accuracy: \t {0:.4f}".format(np.mean(res)))
        print("Accuracy SD: \t\t {0:.4f}".format(np.std(res)))
        
    elif train==False:
        print("Test Result:\n")        
        print("accuracy score: {0:.4f}\n".format(accuracy_score(y_test, clf.predict(X_test))))
        print("Classification Report: \n {}\n".format(classification_report(y_test, clf.predict(X_test))))
        print("Confusion Matrix: \n {}\n".format(confusion_matrix(y_test, clf.predict(X_test))))        

In [ ]:
print_score(knn, X_train, y_train, X_test, y_test, train=True)

In [ ]:
print_score(knn, X_train, y_train, X_test, y_test, train=False)

# Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
knn.get_params()

In [ ]:
params = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}

In [ ]:
grid_search_cv = GridSearchCV(KNeighborsClassifier(),
                              params, 
                              n_jobs=-1,
                              verbose=1)

In [ ]:
grid_search_cv.fit(X_train, y_train)

In [ ]:
grid_search_cv.best_estimator_

In [ ]:
print_score(grid_search_cv, X_train, y_train, X_test, y_test, train=True)

In [ ]:
print_score(grid_search_cv, X_train, y_train, X_test, y_test, train=False)

In [ ]:
grid_search_cv.best_params_

In [ ]:
grid_search_cv.cv_results_['mean_train_score']

In [ ]:
grid_search_cv.cv_results_

SVM, Random Forest, XGBoost

In [ ]:
from sklearn import svm
clf = svm.SVC(kernel='rbf')
clf.fit(X_train, y_train)
print_score(clf, X_train, y_train, X_test, y_test, train=True)
print_score(clf, X_train, y_train, X_test, y_test, train=False)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
print_score(clf, X_train, y_train, X_test, y_test, train=True)
print_score(clf, X_train, y_train, X_test, y_test, train=False)

In [ ]:
import xgboost as xgb
clf = xgb.XGBClassifier()
clf.fit(X_train, y_train)
print_score(clf, X_train, y_train, X_test, y_test, train=True)
print_score(clf, X_train, y_train, X_test, y_test, train=False)

***